MERGING IOA/VITALS AND MED

In [71]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="Workbook contains no default style"
) ### STOP WARNINGS OPENPYXL

In [72]:
import pandas as pd
import numpy as np
import re

In [73]:


# ============================================================
# 1. LOADING FILES
# ============================================================
# Force 'nda' to string to avoid type mismatch during merges
df_med = pd.read_csv('df_med_subset22pel.csv', dtype={'nda': str})
df_ioa_vital = pd.read_csv('df_ioafile_paramvit_clean_22pel.csv', dtype={'nda': str})
df_admin = pd.read_csv('df_admin_subset22_pel_sa.csv', dtype={'nda': str})

# ============================================================
# Strict NDA cleaning (removes .0, spaces, leading zeros)
# ============================================================
def clean_nda(series):
    return (
        series.astype(str)
              .str.replace(r'\.0$', '', regex=True)
              .str.strip()
              .str.lstrip('0')
    )

print("🔧 Cleaning NDA formats...")
df_med['nda'] = clean_nda(df_med['nda'])
df_ioa_vital['nda'] = clean_nda(df_ioa_vital['nda'])

# ============================================================
# 2. FULL OUTER MERGE
# ============================================================
# Full merge to identify orphan records
df_merged = pd.merge(
    df_ioa_vital,
    df_med,
    on='nda',
    how='outer',
    indicator='merge_check'
)

# ============================================================
# 3. MERGE QUALITY REPORT
# ============================================================
stats = df_merged['merge_check'].value_counts()

mapping_bilan = {
    'both': '✅ Complete record (IOA + Medical)',
    'left_only': '⚠️ IOA only (No medical record)',
    'right_only': '⚠️ Medical only (No IOA record)'
}

bilan_df = pd.DataFrame({
    'Number of records': stats.values,
    'Percentage (%)': (stats.values / len(df_merged) * 100).round(2)
}, index=stats.index.map(mapping_bilan))

print("📊 FULL MERGE QUALITY REPORT")
print("-" * 50)
display(bilan_df)
print("-" * 50)

# ============================================================
# 4. EXPORT ANOMALY COUNTS
# ============================================================
missing_med_count = len(df_merged[df_merged['merge_check'] == 'left_only'])
missing_ioa_count = len(df_merged[df_merged['merge_check'] == 'right_only'])

print(f"Total merged rows: {len(df_merged)}")
print(f"Patients without medical record: {missing_med_count}")
print(f"Patients with medical record but no IOA entry: {missing_ioa_count}")


🔧 Cleaning NDA formats...
📊 FULL MERGE QUALITY REPORT
--------------------------------------------------


,Number of records,Percentage (%)
merge_check,,
✅ Complete record (IOA + Medical),29876,68.81
⚠️ IOA only (No medical record),10812,24.90
⚠️ Medical only (No IOA record),2727,6.28


--------------------------------------------------
Total merged rows: 43415
Patients without medical record: 10812
Patients with medical record but no IOA entry: 2727


### Investigation of patient records with IOA but no medical record (potentially lost for clustering) or the opposite (medical record but no IOA° ###

In [74]:
import pandas as pd

# ============================================================
# 1. HARMONIZATION AND MERGE
# ============================================================
# Ensure NDA formats match before merging
df_ioa_vital['nda'] = (
    df_ioa_vital['nda'].astype(str)
                       .str.replace(r'\.0$', '', regex=True)
                       .str.strip()
                       .str.lstrip('0')
)

df_med['nda'] = (
    df_med['nda'].astype(str)
                 .str.replace(r'\.0$', '', regex=True)
                 .str.strip()
                 .str.lstrip('0')
)

df_diag = pd.merge(
    df_ioa_vital,
    df_med,
    on='nda',
    how='left',
    indicator='merge_check'
)

# ============================================================
# 2. FILTER IOA-ONLY RECORDS
# ============================================================
# Keep only rows with no matching medical record
perdus = df_diag[df_diag['merge_check'] == 'left_only'].copy()

# ============================================================
# 3. DATE PREPARATION
# ============================================================
# Ensure date column is properly formatted
perdus['date_adm_final'] = pd.to_datetime(perdus['date_adm_final'])
perdus['mois'] = perdus['date_adm_final'].dt.to_period('M')

# ============================================================
# 4. MONTHLY DISTRIBUTION OF MISSING MEDICAL RECORDS
# ============================================================
print("📅 MONTHLY DISTRIBUTION OF PATIENTS WITHOUT MEDICAL RECORD")
print("-" * 60)

repartition_perte = perdus['mois'].value_counts().sort_index()
total_perte = len(perdus)

df_repartition = pd.DataFrame({
    'Missing Patients Count': repartition_perte,
    '% of Total Missing': ((repartition_perte / total_perte) * 100).round(1)
})

display(df_repartition)

# ============================================================
# 5. COMPARISON WITH TOTAL MONTHLY VOLUME
# ============================================================
print("\n💡 ANALYSIS:")

# Compare missing rate to total IOA volume per month
volume_global = (
    df_ioa_vital['date_adm_final']
    .astype('datetime64[ns]')
    .dt.to_period('M')
    .value_counts()
    .sort_index()
)

taux_perte_mensuel = (repartition_perte / volume_global * 100).round(1)

print("Missing rate relative to total monthly volume:")
print(taux_perte_mensuel)


📅 MONTHLY DISTRIBUTION OF PATIENTS WITHOUT MEDICAL RECORD
------------------------------------------------------------


,Missing Patients Count,% of Total Missing
mois,,
2022-01,876,8.1
2022-02,912,8.4
2022-03,1023,9.5
2022-04,1116,10.3
2022-05,892,8.3
2022-06,789,7.3
2022-07,858,7.9
2022-08,895,8.3
2022-09,837,7.7



💡 ANALYSIS:
Missing rate relative to total monthly volume:
mois
2022-01    23.8
2022-02    27.0
2022-03    25.6
2022-04    29.4
2022-05    26.8
2022-06    25.8
2022-07    27.1
2022-08    27.6
2022-09    25.9
2022-10    26.9
2022-11    26.8
2022-12    26.0
Freq: M, Name: count, dtype: float64


In [75]:
import pandas as pd

# ============================================================
# 1. LOADING AND STRICT CLEANING
# ============================================================
def get_clean_nda_set(df, col_name):
    return set(
        df[col_name]
        .dropna()
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .str.strip()
        .str.lstrip('0')
    )

# Load the three datasets
df_med = pd.read_csv('df_med_subset22pel.csv')
df_ioa_vital = pd.read_csv('df_ioafile_paramvit_clean_22pel.csv')
df_admin = pd.read_csv('df_admin_subset22_pel_sa.csv')

# Extract cleaned NDA sets
set_admin = get_clean_nda_set(df_admin, 'nda')
set_ioa = get_clean_nda_set(df_ioa_vital, 'nda')
set_med = get_clean_nda_set(df_med, 'nda')

# ============================================================
# 2. SET INTERSECTIONS (3-WAY CONSISTENCY CHECK)
# ============================================================
all_three = set_admin.intersection(set_ioa).intersection(set_med)
admin_and_ioa_no_med = set_admin.intersection(set_ioa).difference(set_med)
admin_only = set_admin.difference(set_ioa).difference(set_med)
med_only = set_med.difference(set_admin).difference(set_ioa)

# ============================================================
# 3. TRIPARTITE CONSISTENCY REPORT
# ============================================================
print("🔍 DATASET CONSISTENCY AUDIT (3 SOURCES)")
print("-" * 50)
print(f"📁 ADMIN source (total population): {len(set_admin)}")
print(f"📁 IOA source (vital signs):       {len(set_ioa)}")
print(f"📁 MEDICAL source (diagnoses):     {len(set_med)}")
print("-" * 50)

print(f"✅ COMPLETE RECORDS (3/3):          {len(all_three)}")
print(f"⚠️ ADMIN + IOA but NO MEDICAL:      {len(admin_and_ioa_no_med)}")
print(f"🚫 ADMIN only (total loss):         {len(admin_only)}")
print(f"❓ MEDICAL orphan (no admin):       {len(med_only)}")
print("-" * 50)

# ============================================================
# 4. COMPLETENESS RATE
# ============================================================
if len(set_admin) > 0:
    completeness_rate = (len(all_three) / len(set_admin)) * 100
    print(f"📊 Usable records for clustering: {completeness_rate:.1f}%")


🔍 DATASET CONSISTENCY AUDIT (3 SOURCES)
--------------------------------------------------
📁 ADMIN source (total population): 68266
📁 IOA source (vital signs):       40688
📁 MEDICAL source (diagnoses):     32603
--------------------------------------------------
✅ COMPLETE RECORDS (3/3):          29876
⚠️ ADMIN + IOA but NO MEDICAL:      10805
🚫 ADMIN only (total loss):         24859
❓ MEDICAL orphan (no admin):       1
--------------------------------------------------
📊 Usable records for clustering: 43.8%


In [76]:
# ===============================
# STRICT ADMIN FILTERING (PELLEGRIN)
# To be removed once full dataset is available
# ===============================

# 1. Load the full administrative merged file
df_admin_full = pd.read_csv('df_admin_subset22_pel_sa.csv')

# 2. Safety cleaning of the service column
# Convert to string, remove ".0", strip invisible spaces
df_admin_full['uam_service'] = (
    df_admin_full['uam_service']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
)

# 3. Apply Pellegrin filter (9780)
print(f"📊 Volume before filtering: {len(df_admin)} rows")

df_admin = df_admin_full[df_admin_full['uam_service'] == '9780'].copy()

print(f"✅ Volume after Pellegrin filtering (9780): {len(df_admin)} rows")

# 4. Save the cleaned file for clustering
df_admin.to_csv('df_admin_2022_PELLEGRIN_clean.csv', index=False)

# 5. Quick check
if not df_admin.empty:
    print("\nPreview of the first 5 filtered rows:")
    display(df_admin[['nda', 'uam_service', 'date_entree']].head())
else:
    print("\n⚠️ WARNING: The filter removed all rows. Check the exact service code or column name.")
    print("Unique values found in the column:", df_admin_full['uam_service'].unique()[:10])


📊 Volume before filtering: 68266 rows
✅ Volume after Pellegrin filtering (9780): 48042 rows

Preview of the first 5 filtered rows:


,nda,uam_service,date_entree
1,22030011617,9780,2022-01-01 00:08:00
2,22030011619,9780,2022-01-01 00:17:00
4,22030011622,9780,2022-01-01 00:22:00
5,22030011625,9780,2022-01-01 00:31:00
6,22030011629,9780,2022-01-01 00:35:00


In [77]:
# ============================================================
# 1. UNIFIED NDA CLEANING FUNCTION
# ============================================================
def clean_nda(series):
    """Clean NDA identifiers: remove .0, trim spaces, remove leading zeros."""
    return (
        series.dropna()
              .astype(str)
              .str.replace(r'\.0$', '', regex=True)
              .str.strip()
              .str.lstrip('0')
    )

# ============================================================
# 2. LOAD ALL SOURCES
# ============================================================
print("Loading datasets...")

d_admin = pd.read_csv('df_admin_2022_PELLEGRIN_clean.csv')
d_ioa = pd.read_csv('df_ioa_subset22pel.csv')
d_med = pd.read_csv('df_med_subset22pel.csv')
d_vits = pd.read_csv('df_param_vit_subset22pel.csv')

# ============================================================
# 3. CLEAN NDA AND BUILD SETS
# ============================================================
sources = {
    "ADMIN": clean_nda(d_admin['nda']),
    "IOA": clean_nda(d_ioa['nda']),
    "VITALS": clean_nda(d_vits['nda']),
    "MEDICAL": clean_nda(d_med['nda'])
}

sets = {name: set(values) for name, values in sources.items()}

# ============================================================
# 4. GLOBAL COMPLETENESS (4-WAY INTERSECTION)
# ============================================================
all_common = (
    sets["ADMIN"]
    .intersection(sets["IOA"])
    .intersection(sets["VITALS"])
    .intersection(sets["MEDICAL"])
)

print("\n" + "=" * 70)
print("🔎 GLOBAL NDA COMPLETENESS AUDIT (4 SOURCES)")
print("=" * 70)

for name, s in sets.items():
    print(f"📁 {name:<15} : {len(s):>6} unique patients")

print("-" * 70)
print(f"✅ COMPLETE PATIENTS (4/4) : {len(all_common)}")
print(f"📉 COMPLETENESS RATE       : {(len(all_common) / len(sets['ADMIN']) * 100):.1f}%")
print("-" * 70)

# ============================================================
# 5. DETAILED MISSINGNESS ANALYSIS (PAIRWISE GAPS)
# ============================================================
ioa_no_vits = len(sets["IOA"] - sets["VITALS"])
ioa_no_med = len(sets["IOA"] - sets["MEDICAL"])
med_no_ioa = len(sets["MEDICAL"] - sets["IOA"])
admin_no_med = len(sets["ADMIN"] - sets["MEDICAL"])
admin_no_ioa = len(sets["ADMIN"] - sets["IOA"])
vits_no_admin = len(sets["VITALS"] - sets["ADMIN"])

print("\n" + "=" * 70)
print("🩺 DETAILED DISCORDANCE ANALYSIS (ORPHAN RECORDS)")
print("=" * 70)

print(f"🔹 IOA but NO VITALS                 = {ioa_no_vits}")
print(f"🔹 IOA but NO MEDICAL                = {ioa_no_med}")
print(f"🔹 MEDICAL but NO IOA                = {med_no_ioa}")
print(f"🔹 ADMIN but NO MEDICAL              = {admin_no_med}")
print(f"🔹 ADMIN but NO IOA                  = {admin_no_ioa}")
print(f"🔹 VITALS but NO ADMIN               = {vits_no_admin}")

print("-" * 70)
print(f"✅ FULLY USABLE RECORDS (4/4)        : {len(all_common)}")
print("=" * 70)

# ============================================================
# 6. COLUMN INVENTORY FOR ALL FILES
# ============================================================
files = {
    "ADMIN": "df_admin_2022_PELLEGRIN_clean.csv",
    "MEDICAL": "df_med_subset22pel.csv",
    "IOA/VITALS": "df_ioafile_paramvit_clean_22pel.csv"
}

print("\n📋 COLUMN INVENTORY BY SOURCE")
print("=" * 50)

all_columns = {}

for name, path in files.items():
    try:
        df_temp = pd.read_csv(path, nrows=1)
        cols = df_temp.columns.tolist()
        all_columns[name] = cols

        print(f"\n📂 Source: {name}")
        print(f"Total columns: {len(cols)}")
        print(f"List: {', '.join(cols)}")
        print("-" * 50)

    except Exception as e:
        print(f"❌ Could not read {name}: {e}")

# ============================================================
# 7. COMMON COLUMNS ACROSS ALL FILES
# ============================================================
if len(all_columns) > 1:
    common_cols = set.intersection(*[set(cols) for cols in all_columns.values()])
    print("\n🔑 Columns common to ALL files:")
    print(common_cols if common_cols else "None (check join keys!)")


Loading datasets...

🔎 GLOBAL NDA COMPLETENESS AUDIT (4 SOURCES)
📁 ADMIN           :  48042 unique patients
📁 IOA             :  43428 unique patients
📁 VITALS          :  30570 unique patients
📁 MEDICAL         :  32603 unique patients
----------------------------------------------------------------------
✅ COMPLETE PATIENTS (4/4) : 23183
📉 COMPLETENESS RATE       : 48.3%
----------------------------------------------------------------------

🩺 DETAILED DISCORDANCE ANALYSIS (ORPHAN RECORDS)
🔹 IOA but NO VITALS                 = 13787
🔹 IOA but NO MEDICAL                = 11988
🔹 MEDICAL but NO IOA                = 1163
🔹 ADMIN but NO MEDICAL              = 15440
🔹 ADMIN but NO IOA                  = 4621
🔹 VITALS but NO ADMIN               = 7
----------------------------------------------------------------------
✅ FULLY USABLE RECORDS (4/4)        : 23183

📋 COLUMN INVENTORY BY SOURCE

📂 Source: ADMIN
Total columns: 26
List: Unnamed: 0, uam_service, nda, date_entree, nom, prenom, dat

EKG MINING

In [78]:
import pandas as pd
import numpy as np
import re

# ============================================================
# 1. DEFINITION DE LA FONCTION (Pour éviter le NameError)
# ============================================================
def get_context_snippet(row, window=80):
    """Extrait un morceau de texte autour du mot-clé trouvé."""
    try:
        sources_list = str(row['ekg_source_column']).split(' & ')
        primary_col = sources_list[0]

        if primary_col == "not_found" or primary_col not in row:
            return ""

        text_str = str(row[primary_col])
        keyword = str(row['ekg_keyword_found'])

        if not keyword or keyword == 'nan':
            return "Keyword missing"

        match = re.search(re.escape(keyword), text_str, re.IGNORECASE)
        if match:
            start = max(0, match.start() - window)
            end = min(len(text_str), match.end() + window)
            return f"...{text_str[start:end]}..."
        return "Context not found"
    except:
        return "Error in snippet"

# ============================================================
# 2. DETECTION EKG (5 COLONNES)
# ============================================================
cols_to_scan = ['clinical_exam', 'evolution', 'conclusion', 'additional_tests', 'evolution_ioa']
print(f"🔍 Scanning {len(df_med)} records...")

# Nettoyage et normalisation
c_exam  = df_med['clinical_exam'].fillna('').astype(str).str.lower()
c_evol  = df_med['evolution'].fillna('').astype(str).str.lower()
c_conc  = df_med['conclusion'].fillna('').astype(str).str.lower()
c_tests = df_med['additional_tests'].fillna('').astype(str).str.lower()
c_ioa   = df_med['evolution_ioa'].fillna('').astype(str).str.lower() if 'evolution_ioa' in df_med.columns else pd.Series([''] * len(df_med))

ekg_pattern = r'(\becg\b|electrocar|e\.c\.g)'

f_exam  = c_exam.str.contains(ekg_pattern, regex=True)
f_evol  = c_evol.str.contains(ekg_pattern, regex=True)
f_conc  = c_conc.str.contains(ekg_pattern, regex=True)
f_tests = c_tests.str.contains(ekg_pattern, regex=True)
f_ioa   = c_ioa.str.contains(ekg_pattern, regex=True)

# Création du flag final dans df_med
df_med['had_ekg'] = (f_exam | f_evol | f_conc | f_tests | f_ioa).astype(int)

# Identification des sources
def identify_sources(row_idx):
    srcs = []
    if f_exam[row_idx]:  srcs.append('clinical_exam')
    if f_evol[row_idx]:  srcs.append('evolution')
    if f_conc[row_idx]:  srcs.append('conclusion')
    if f_tests[row_idx]: srcs.append('additional_tests')
    if f_ioa[row_idx]:   srcs.append('evolution_ioa')
    return " & ".join(srcs) if srcs else "not_found"

df_med['ekg_source_column'] = [identify_sources(i) for i in range(len(df_med))]

# Extraction du mot-clé
all_text = c_exam + " " + c_evol + " " + c_conc + " " + c_tests + " " + c_ioa
df_med['ekg_keyword_found'] = all_text.str.extract(ekg_pattern, expand=False)

# Stats IOA
med_found = (f_exam | f_evol | f_conc | f_tests)
ioa_only_count = (f_ioa & ~med_found).sum()

# ============================================================
# 3. EXPORT AUDIT
# ============================================================
df_ekg_all = df_med[df_med['had_ekg'] == 1].copy()

if not df_ekg_all.empty:
    df_electrocar = df_ekg_all[df_ekg_all['ekg_keyword_found'].str.contains('electrocar', na=False)]
    df_others = df_ekg_all[~df_ekg_all['ekg_keyword_found'].str.contains('electrocar', na=False)]

    sample_size = min(300, len(df_others))
    df_sample_others = df_others.sample(n=sample_size, random_state=42)
    df_audit = pd.concat([df_electrocar, df_sample_others]).drop_duplicates(subset='nda').copy()

    print("🪄 Generating context snippets...")
    # On applique la fonction définie plus haut
    df_audit['ekg_snippet'] = df_audit.apply(get_context_snippet, axis=1)

    output_file = "audit_ekg_final_with_counts.csv"
    export_cols = ['nda', 'diag', 'ekg_source_column', 'ekg_keyword_found', 'ekg_snippet']
    df_audit[export_cols].to_csv(output_file, index=False, sep=';', encoding='utf-8-sig')
    print(f"✅ Audit file created: {output_file}")
else:
    print("⚠️ No EKG detected.")

print("\n" + "="*45)
print(f"📊 SUMMARY: {df_med['had_ekg'].sum()} total EKGs.")
print(f"💎 IOA-ONLY: {ioa_only_count}")
print("="*45)

🔍 Scanning 32603 records...


/tmp/ipykernel_2811938/1614411695.py:47: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f_exam  = c_exam.str.contains(ekg_pattern, regex=True)
/tmp/ipykernel_2811938/1614411695.py:48: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f_evol  = c_evol.str.contains(ekg_pattern, regex=True)
/tmp/ipykernel_2811938/1614411695.py:49: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f_conc  = c_conc.str.contains(ekg_pattern, regex=True)
/tmp/ipykernel_2811938/1614411695.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f_tests = c_tests.str.contains(ekg_pattern, regex=True)
/tmp/ipykernel_2811938/1614411695.py:51: UserWarning: This pattern is inter

🪄 Generating context snippets...
✅ Audit file created: audit_ekg_final_with_counts.csv

📊 SUMMARY: 8616 total EKGs.
💎 IOA-ONLY: 0


In [79]:
# import pandas as pd
# import numpy as np
# import re
#
# # ============================================================
# # 1. ROBUST EKG DETECTION (5 COLUMNS: 4 MEDICAL + 1 IOA)
# # ============================================================
# # We scan 4 doctor columns + 1 nurse column (evolution_ioa)
# cols_to_scan = ['clinical_exam', 'evolution', 'conclusion', 'additional_tests', 'evolution_ioa']
# print(f"🔍 Scanning {len(df_med)} records across medical and IOA columns...")
#
# # 1. Clean and normalize text columns
# c_exam  = df_med['clinical_exam'].fillna('').astype(str).str.lower()
# c_evol  = df_med['evolution'].fillna('').astype(str).str.lower()
# c_conc  = df_med['conclusion'].fillna('').astype(str).str.lower()
# c_tests = df_med['additional_tests'].fillna('').astype(str).str.lower()
# # Safety check for IOA column
# c_ioa   = df_med['evolution_ioa'].fillna('').astype(str).str.lower() if 'evolution_ioa' in df_med.columns else pd.Series([''] * len(df_med))
#
# # 2. Define Regex Pattern (Matches 'ecg', 'electrocar...', 'e.c.g')
# ekg_pattern = r'(\becg\b|electrocar|e\.c\.g)'
#
# # 3. Create individual flags (Boolean masks)
# f_exam  = c_exam.str.contains(ekg_pattern, regex=True)
# f_evol  = c_evol.str.contains(ekg_pattern, regex=True)
# f_conc  = c_conc.str.contains(ekg_pattern, regex=True)
# f_tests = c_tests.str.contains(ekg_pattern, regex=True)
# f_ioa   = c_ioa.str.contains(ekg_pattern, regex=True)
#
# # 4. Create Global Binary Flag 'had_ekg' (Logical OR)
# # If found in ANY of the 5 columns, it's a 1
# df_med['had_ekg'] = (f_exam | f_evol | f_conc | f_tests | f_ioa).astype(int)
#
# # 5. Identify all sources (Cumulative: e.g., "clinical_exam & evolution_ioa")
# def identify_sources(row_idx):
#     srcs = []
#     if f_exam[row_idx]:  srcs.append('clinical_exam')
#     if f_evol[row_idx]:  srcs.append('evolution')
#     if f_conc[row_idx]:  srcs.append('conclusion')
#     if f_tests[row_idx]: srcs.append('additional_tests')
#     if f_ioa[row_idx]:   srcs.append('evolution_ioa')
#     return " & ".join(srcs) if srcs else "not_found"
#
# df_med['ekg_source_column'] = [identify_sources(i) for i in range(len(df_med))]
#
# # 6. Extract the specific keyword found for audit stats
# # Combine all texts to extract the word that triggered the match
# all_text_combined = c_exam + " " + c_evol + " " + c_conc + " " + c_tests + " " + c_ioa
# df_med['ekg_keyword_found'] = all_text_combined.str.extract(ekg_pattern, expand=False)
#
# # ============================================================
# # 2. STATISTICS: THE "IOA GAIN"
# # ============================================================
# # Count patients found ONLY in IOA notes and NOT in medical notes
# med_found = (f_exam | f_evol | f_conc | f_tests)
# ioa_only_count = (f_ioa & ~med_found).sum()
#
# # ============================================================
# # 3. AUDIT SAMPLE CREATION & EXPORT
# # ============================================================
# df_ekg_all = df_med[df_med['had_ekg'] == 1].copy()
#
# if not df_ekg_all.empty:
#     # We keep all 'electrocar' and sample 300 of the others
#     df_electrocar = df_ekg_all[df_ekg_all['ekg_keyword_found'].str.contains('electrocar', na=False)]
#     df_others = df_ekg_all[~df_ekg_all['ekg_keyword_found'].str.contains('electrocar', na=False)]
#
#     sample_size = min(300, len(df_others))
#     df_sample_others = df_others.sample(n=sample_size, random_state=42)
#     df_audit = pd.concat([df_electrocar, df_sample_others]).drop_duplicates(subset='nda').copy()
#
#     # Apply snippet function (defined in your previous block)
#     print("🪄 Generating context snippets for audit...")
#     df_audit['ekg_snippet'] = df_audit.apply(get_context_snippet, axis=1)
#
#     # Save to CSV
#     output_file = "audit_ekg_final_with_counts.csv"
#     export_cols = ['nda', 'diag', 'ekg_source_column', 'ekg_keyword_found', 'ekg_snippet']
#     df_audit[export_cols].to_csv(output_file, index=False, sep=';', encoding='utf-8-sig')
#     print(f"✅ Audit file created: {output_file}")
# else:
#     print("⚠️ No EKG detected.")
#
# # ============================================================
# # 4. FINAL SUMMARY
# # ============================================================
# print("\n" + "="*45)
# print(f"📊 FINAL SUMMARY")
# print(f"Total Patients with EKG (had_ekg): {df_med['had_ekg'].sum()}")
# print(f"💎 IOA-ONLY Detections: {ioa_only_count}")
# print(f"   (Found in IOA notes but missing in Medical notes)")
# print("="*45)

In [80]:
# import pandas as pd
# import numpy as np
# import re
#
# # ============================================================
# # 1. ROBUST EKG DETECTION (4 COLUMNS & MULTI-SOURCE)
# # ============================================================
# print(f"🔍 Scanning {len(df_med)} records across 4 medical columns...")
#
# # 1. Clean and normalize text columns
# c_exam  = df_med['clinical_exam'].fillna('').astype(str).str.lower()
# c_evol  = df_med['evolution'].fillna('').astype(str).str.lower()
# c_conc  = df_med['conclusion'].fillna('').astype(str).str.lower()
# c_tests = df_med['additional_tests'].fillna('').astype(str).str.lower()
#
# # 2. Define Regex Pattern
# ekg_pattern = r'(\becg\b|electrocar|e\.c\.g)'
#
# # 3. Create individual flags for each column
# f_exam  = c_exam.str.contains(ekg_pattern, regex=True)
# f_evol  = c_evol.str.contains(ekg_pattern, regex=True)
# f_conc  = c_conc.str.contains(ekg_pattern, regex=True)
# f_tests = c_tests.str.contains(ekg_pattern, regex=True)
#
# # 4. Create Global Binary Flag (Renamed to HAD_EKG for consistency)
# df_med['had_ekg'] = (f_exam | f_evol | f_conc | f_tests).astype(int)
#
# # 5. Identify all sources
# def identify_sources(row_idx):
#     srcs = []
#     if f_exam[row_idx]:  srcs.append('clinical_exam')
#     if f_evol[row_idx]:  srcs.append('evolution')
#     if f_conc[row_idx]:  srcs.append('conclusion')
#     if f_tests[row_idx]: srcs.append('additional_tests')
#     return " & ".join(srcs) if srcs else "not_found"
#
# df_med['ekg_source_column'] = [identify_sources(i) for i in range(len(df_med))]
#
# # 6. Extract the specific keyword found
# combined_text = c_exam + " " + c_evol + " " + c_conc + " " + c_tests
# df_med['ekg_keyword_found'] = combined_text.str.extract(ekg_pattern, expand=False)
#
# # ============================================================
# # 2. CONTEXT SNIPPET FUNCTION (FOR AUDIT)
# # ============================================================
# def get_context_snippet(row, window=80):
#     sources_list = row['ekg_source_column'].split(' & ')
#     primary_col = sources_list[0]
#
#     if primary_col == "not_found": return ""
#
#     text_str = str(row[primary_col])
#     keyword = str(row['ekg_keyword_found'])
#
#     if not keyword or keyword == 'nan': return "Keyword missing"
#
#     match = re.search(re.escape(keyword), text_str, re.IGNORECASE)
#     if match:
#         start = max(0, match.start() - window)
#         end = min(len(text_str), match.end() + window)
#         return f"...{text_str[start:end]}..."
#     return "Context not found"
#
# # ============================================================
# # 3. AUDIT SAMPLE CREATION & EXPORT
# # ============================================================
# print("📊 Preparing audit sample...")
#
# # Using 'had_ekg' here
# df_ekg_all = df_med[df_med['had_ekg'] == 1].copy()
#
# if len(df_ekg_all) > 0:
#     df_electrocar = df_ekg_all[df_ekg_all['ekg_keyword_found'].str.contains('electrocar', na=False)]
#     df_others = df_ekg_all[~df_ekg_all['ekg_keyword_found'].str.contains('electrocar', na=False)]
#
#     sample_size = min(300, len(df_others))
#     df_sample_others = df_others.sample(n=sample_size, random_state=42)
#
#     df_audit = pd.concat([df_electrocar, df_sample_others]).drop_duplicates(subset='nda').copy()
#
#     print("🪄 Generating context snippets...")
#     df_audit['ekg_snippet'] = df_audit.apply(get_context_snippet, axis=1)
#
#     output_file = "audit_ekg_final_with_counts.csv"
#     export_cols = ['nda', 'diag', 'ekg_source_column', 'ekg_keyword_found', 'ekg_snippet']
#     df_audit[export_cols].to_csv(output_file, index=False, sep=';', encoding='utf-8-sig')
#
#     print(f"✅ Audit file created: {output_file}")
# else:
#     print("⚠️ No EKG detected.")
#
# # ============================================================
# # 4. FINAL SUMMARY
# # ============================================================
# print("\n" + "="*40)
# print(f"FINAL SUMMARY: {df_med['had_ekg'].sum()} total EKGs (had_ekg) detected.")
# print("="*40)

### ACTUAL MERGING OF THE 3 FILES (IOA/VITALS + ADMIN + MEDICAL) ###


In [81]:
import pandas as pd
import numpy as np
import re

# ============================================================
# 1. FUNCTION DEFINITION (MUST BE DEFINED BEFORE USE)
# ============================================================
def get_context_snippet(row, window=80):
    """ Extracts text around the keyword from the primary source found. """
    if 'ekg_source_column' not in row or 'ekg_keyword_found' not in row:
        return "Missing tracking columns"

    sources_list = str(row['ekg_source_column']).split(' & ')
    primary_col = sources_list[0]

    if primary_col == "not_found" or primary_col not in row.index:
        return ""

    text_str = str(row[primary_col])
    keyword = str(row['ekg_keyword_found'])

    if not keyword or keyword == 'nan':
        return "Keyword missing"

    # Search for the keyword position
    match = re.search(re.escape(keyword), text_str, re.IGNORECASE)
    if match:
        start = max(0, match.start() - window)
        end = min(len(text_str), match.end() + window)
        return f"...{text_str[start:end]}..."

    return "Context not found"

# ============================================================
# 2. ROBUST EKG DETECTION (5 COLUMNS: 4 MEDICAL + 1 IOA)
# ============================================================
print(f"🔍 Scanning {len(df_med)} records...")

# 1. Clean and normalize
c_exam  = df_med['clinical_exam'].fillna('').astype(str).str.lower()
c_evol  = df_med['evolution'].fillna('').astype(str).str.lower()
c_conc  = df_med['conclusion'].fillna('').astype(str).str.lower()
c_tests = df_med['additional_tests'].fillna('').astype(str).str.lower()
c_ioa   = df_med['evolution_ioa'].fillna('').astype(str).str.lower() if 'evolution_ioa' in df_med.columns else pd.Series([''] * len(df_med))

# 2. Regex Pattern (Non-capturing group to avoid warnings)
ekg_pattern = r'(?:\becg\b|electrocar|e\.c\.g)'

# 3. Flags
f_exam = c_exam.str.contains(ekg_pattern, regex=True)
f_evol = c_evol.str.contains(ekg_pattern, regex=True)
f_conc = c_conc.str.contains(ekg_pattern, regex=True)
f_tests = c_tests.str.contains(ekg_pattern, regex=True)
f_ioa = c_ioa.str.contains(ekg_pattern, regex=True)

# 4. Global Binary Flag
df_med['had_ekg'] = (f_exam | f_evol | f_conc | f_tests | f_ioa).astype(int)

# 5. Sources identification
def identify_sources(row_idx):
    srcs = []
    if f_exam[row_idx]:  srcs.append('clinical_exam')
    if f_evol[row_idx]:  srcs.append('evolution')
    if f_conc[row_idx]:  srcs.append('conclusion')
    if f_tests[row_idx]: srcs.append('additional_tests')
    if f_ioa[row_idx]:   srcs.append('evolution_ioa')
    return " & ".join(srcs) if srcs else "not_found"

df_med['ekg_source_column'] = [identify_sources(i) for i in range(len(df_med))]
df_med['ekg_keyword_found'] = (c_exam + " " + c_evol + " " + c_conc + " " + c_tests + " " + c_ioa).str.extract(f'({ekg_pattern})', expand=False)

# ============================================================
# 3. AUDIT SAMPLE CREATION & EXPORT
# ============================================================
df_ekg_all = df_med[df_med['had_ekg'] == 1].copy()

if not df_ekg_all.empty:
    print("🪄 Generating context snippets for audit...")
    # Sampling for manageable audit
    df_audit = df_ekg_all.sample(n=min(500, len(df_ekg_all)), random_state=42).copy()

    # NOW THE FUNCTION IS DEFINED IN THE SAME BLOCK
    df_audit['ekg_snippet'] = df_audit.apply(get_context_snippet, axis=1)

    output_file = "audit_ekg_final_with_counts.csv"
    export_cols = ['nda', 'diag', 'ekg_source_column', 'ekg_keyword_found', 'ekg_snippet']
    df_audit[export_cols].to_csv(output_file, index=False, sep=';', encoding='utf-8-sig')
    print(f"✅ Audit file created: {output_file}")
else:
    print("⚠️ No EKG detected.")

# Final Summary
print(f"\n📊 TOTAL EKGs (had_ekg): {df_med['had_ekg'].sum()}")

🔍 Scanning 32603 records...
🪄 Generating context snippets for audit...
✅ Audit file created: audit_ekg_final_with_counts.csv

📊 TOTAL EKGs (had_ekg): 8616


In [82]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD SOURCES
# ============================================================
print("📥 Loading datasets...")

df_admin = pd.read_csv("df_admin_subset22pel.csv")
df_ioa = pd.read_csv("df_ioafile_paramvit_clean_22pel.csv")


# ============================================================
# 2. NDA CLEANING (STRICT MATCHING)
# ============================================================
for df in [df_admin, df_ioa, df_med]:
    df['nda'] = (
        df['nda'].astype(str)
                 .str.replace(r'\.0$', '', regex=True)
                 .str.strip()
                 .str.lstrip('0')
    )

# ============================================================
# 3. COLUMN SELECTION AND RENAMING
# ============================================================

# Admin subset
df_admin_sub = df_admin[['nda', 'uam_service', 'date_entree', 'date_sortie', 'date_naissance', 'mode_sortie', 'decision_urgence']].rename(
    columns={'date_entree': 'date_entree_adm_file'}
)

# Medical subset
cols_med = [
    'nda', 'sex', 'age', 'date_adm_med', 'diag', 'anam_ed', 'atcd_med', 'had_ekg', 'additional_tests',
    'rx_home', 'clinical_exam', 'rx_ed', 'evolution', 'conclusion', 'disposition', 'source_file'
]

df_med_sub = df_med[cols_med].rename(
    columns={'date_adm_med': 'date_adm_med_file', 'anam_urg': 'anam_med'}
)

# ============================================================
# 4. MERGING PIPELINE (INNER MERGE = COMPLETE RECORDS ONLY)
# ============================================================
print("⚙️ Merging datasets...")

# Step A: Admin + IOA
inter_admin_ioa = pd.merge(df_admin_sub, df_ioa, on='nda', how='inner')
lost_ioa = len(df_admin_sub) - len(inter_admin_ioa)

# Step B: Add Medical
df_final = pd.merge(inter_admin_ioa, df_med_sub, on='nda', how='inner')
lost_med = len(inter_admin_ioa) - len(df_final)

# # ============================================================
# # 4.5 CONSOLIDATE ECG FLAGS (ekg_flag + has_ekg)
# # ============================================================
# print("⚡ Consolidating ECG flags...")
#
# # check that variables present and numerical
# for col in ['ekg_flag', 'has_ekg']:
#     if col in df_final.columns:
#         df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0).astype(int)
#     else:
#         df_final[col] = 0 # security if missing column
#
# # creation 'had_ekg' (Logical OR)
# df_final['had_ekg'] = df_final[['ekg_flag', 'has_ekg']].max(axis=1)
#
# # remove old columns
# df_final.drop(columns=['ekg_flag', 'has_ekg'], inplace=True)
#
# print(f"✅ ECG flags unified. Total EKG: {df_final['had_ekg'].sum()}")

# ============================================================
# 5. MERGE REPORT
# ============================================================
print("\n" + "="*60)
print("📊 MERGE REPORT (ONLY COMPLETE RECORDS RETAINED)")
print("="*60)
print(f"✅ Final number of retained patients : {len(df_final)}")
print("-" * 60)
print(f"❌ Removed due to missing IOA/Vitals : {lost_ioa}")
print(f"❌ Removed due to missing Medical    : {lost_med}")
print(f"📉 Total removed                     : {lost_ioa + lost_med}")
print("=" * 60)

# ============================================================
# 6. DATE CONSISTENCY CHECK
# ============================================================
date_cols = ["date_entree_adm_file", "date_adm_final", "date_adm_med_file"]

for col in date_cols:
    df_final[col] = pd.to_datetime(df_final[col], errors='coerce')

check_admin = (df_final["date_entree_adm_file"] == df_final["date_adm_final"]).all()
check_med = (df_final["date_entree_adm_file"] == df_final["date_adm_med_file"]).all()

print("\n🧐 DATE CONSISTENCY CHECK")
print("-" * 40)
print(f"Admin vs Final   : {'✅ Identical' if check_admin else '❌ Different'}")
print(f"Admin vs Medical : {'✅ Identical' if check_med else '❌ Different'}")

if not check_med:
    avg_diff = (df_final["date_adm_med_file"] - df_final["date_entree_adm_file"]).dt.total_seconds().mean() / 60
    print(f"\n💡 Average difference (Admin vs Medical): {avg_diff:.1f} minutes.")

# ============================================================
# 7. UNIFY ADMISSION DATE
# ============================================================
df_final.columns = df_final.columns.str.strip()
df_final['datetime_admission'] = df_final.pop('date_adm_final')
df_final.drop(columns=['date_entree_adm_file', 'date_adm_med_file'], inplace=True, errors='ignore')

print("\n✅ Admission date unified into 'datetime_admission'.")

# ============================================================
# 8. COLUMN REORDERING
# ============================================================
desired_start = [
    'nda', 'sex', 'date_naissance', 'age', 'uam_service',
    'hospital', 'origines_donneees', 'transport', 'datetime_admission'
]

existing_start = [c for c in desired_start if c in df_final.columns]
other_cols = [c for c in df_final.columns if c not in existing_start]

df_final = df_final[existing_start + other_cols]

print("\n✅ Columns successfully reordered.")
print(f"First 9 columns: {df_final.columns[:9].tolist()}")

# ============================================================
# 9. AGE VALIDATION (ADMIN vs CALCULATED)
# ============================================================
df_final['date_naissance'] = pd.to_datetime(df_final['date_naissance'], errors='coerce')
df_final['datetime_admission'] = pd.to_datetime(df_final['datetime_admission'], errors='coerce')

age_calc = (df_final['datetime_admission'] - df_final['date_naissance']).dt.days / 365.25
df_final['age_calcule'] = age_calc.apply(lambda x: int(x) if pd.notnull(x) else None)

if 'age' in df_final.columns:
    df_final['age'] = pd.to_numeric(df_final['age'], errors='coerce')
    diff_age = df_final[df_final['age'] != df_final['age_calcule']]

    print("\n📊 AGE CONSISTENCY ANALYSIS")
    print(f"Matching ages     : {len(df_final) - len(diff_age)}")
    print(f"Age discrepancies : {len(diff_age)}")

    if not diff_age.empty:
        print("\nExample discrepancies:")
        print(diff_age[['nda', 'date_naissance', 'datetime_admission', 'age', 'age_calcule']].head())
else:
    df_final['age'] = df_final['age_calcule']
    print("ℹ️ Age column created from birth date.")

df_final.drop(columns=['age_calcule'], inplace=True)

# ============================================================
# 10. AGE ERROR ANALYSIS (> 1 YEAR)
# ============================================================
age_theo = (df_final['datetime_admission'] - df_final['date_naissance']).dt.days / 365.25
df_final['age'] = pd.to_numeric(df_final['age'], errors='coerce')
df_final['ecart_age'] = df_final['age'] - age_theo

print("\n📊 GLOBAL AGE ERROR STATS")
print(df_final['ecart_age'].describe())
print("-" * 40)

mask_err = df_final['ecart_age'].abs() > 1
df_err = df_final[mask_err]

print(f"🧐 AGE DISCREPANCIES > 1 YEAR: {len(df_err)} ({len(df_err)/len(df_final)*100:.2f}%)")

if not df_err.empty:
    print("\n📈 Error-only stats:")
    print

📥 Loading datasets...
⚙️ Merging datasets...

📊 MERGE REPORT (ONLY COMPLETE RECORDS RETAINED)
✅ Final number of retained patients : 29876
------------------------------------------------------------
❌ Removed due to missing IOA/Vitals : 7561
❌ Removed due to missing Medical    : 10811
📉 Total removed                     : 18372

🧐 DATE CONSISTENCY CHECK
----------------------------------------
Admin vs Final   : ✅ Identical
Admin vs Medical : ✅ Identical

✅ Admission date unified into 'datetime_admission'.

✅ Columns successfully reordered.
First 9 columns: ['nda', 'sex', 'date_naissance', 'age', 'uam_service', 'hospital', 'datetime_admission', 'date_sortie', 'mode_sortie']

📊 AGE CONSISTENCY ANALYSIS
Matching ages     : 29846
Age discrepancies : 30

Example discrepancies:
              nda date_naissance  datetime_admission  age  age_calcule
1290  22030063614     1949-01-15 2022-01-15 17:36:00   73           72
5163  22030211108     1941-02-27 2022-02-27 20:06:00   81           80
737

ue seulement 46 patients avec difference, et vue les erreurs sont enorme on va garder l'age pour l'instant, car souvent les date de naissances a l'admission sont inventees quand on ne connait pas l'identite du patient, et presaue 200 paitents aui ont eu un ecg et aui sont pas prsent dans tous les dosisers


In [83]:
# 6. Saving
df_final.to_csv("df_ioa_vital_med_adm_pel_2022_full.csv", index=False)
print("\n💾 Fichier 'df_ioa_vital_med_adm_pel_2022_full.csv' créé avec succès.")


💾 Fichier 'df_ioa_vital_med_adm_pel_2022_full.csv' créé avec succès.


In [84]:
import pandas as pd

# ============================================================
# 1. LOAD FULL DATASET
# ============================================================
file_full = "df_ioa_vital_med_adm_pel_2022_full.csv"
df_final = pd.read_csv(file_full, dtype={'nda': str}, low_memory=False)

# ============================================================
# 2. EXCLUDE NON‑TABULAR / NARRATIVE COLUMNS
# ============================================================
cols_to_exclude = [
    'atcd_med', 'atcd_ioa', 'anam_ioa', 'anam_ed', 'evolution_ioa', 'admission_summary_ioa', 'rx_home',
    'rx_home', 'clinical_exam', 'rx_ed', 'evolution', 'additional_tests',
    'conclusion', 'source_file', 'date_naissance', 'origine_donnees', 'ecart_age'
]

df_tabular = df_final.drop(columns=[c for c in cols_to_exclude if c in df_final.columns])

# ============================================================
# 3. NDA SAFETY CHECK
# ============================================================
if 'nda' in df_tabular.columns:
    df_tabular['nda'] = df_tabular['nda'].astype(str).str.strip()
    print("✅ Column 'nda' preserved and cleaned.")
else:
    print("⚠️ Warning: 'nda' column is missing from the dataset!")

# ============================================================
# 4. SAVE TABULAR DATASET
# ============================================================
output_file = "df_ioa_vital_med_adm_pel_2022_tabular.csv"
df_tabular.to_csv(output_file, index=False)

print(f"✅ File '{output_file}' created with all relevant tabular columns.")
print(f"Final number of columns: {len(df_tabular.columns)}")


✅ Column 'nda' preserved and cleaned.
✅ File 'df_ioa_vital_med_adm_pel_2022_tabular.csv' created with all relevant tabular columns.
Final number of columns: 63
